# Hide & Seek AI — Train from Scratch
**Multi-Agent Reinforcement Learning with Emergent Tool Use**

PPO · Actor-Critic · GAE · CTDE · Self-Play

---

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from dataclasses import dataclass
from typing import List, Tuple, Dict, Optional
import time
import os
import copy
os.makedirs('data', exist_ok=True)

# ── Hardware auto-detection ──────────────────────────────────────────
print(f"PyTorch {torch.__version__}")
IS_KAGGLE = os.environ.get('KAGGLE_KERNEL_RUN_TYPE') is not None

if torch.cuda.is_available():
    DEVICE = torch.device("cuda:0")
    torch.cuda.set_device(0)
    torch.backends.cudnn.benchmark = True
    NUM_ENVS = 64
    BATCH_SIZE = 2048
    NUM_EPOCHS = 10
    MAX_TRAIN_MINUTES = 0             # no time limit
    MAX_TOTAL_STEPS = 60_000_000      # 60M steps target
    _gn = torch.cuda.get_device_name(0)
    _gc = torch.cuda.device_count()
    print(f"✔ CUDA: {_gn}" + (f"  ({_gc} GPUs)" if _gc > 1 else ""))
    try:
        _t = torch.randn(256, 256, device=DEVICE) @ torch.randn(256, 256, device=DEVICE)
        del _t
        torch.cuda.synchronize()
        print(f"  CUDA test passed ✔")
    except Exception as e:
        print(f"  ⚠ CUDA failed: {e}, falling back to CPU")
        DEVICE = torch.device("cpu")
        NUM_ENVS = 4; BATCH_SIZE = 256; NUM_EPOCHS = 4
        MAX_TRAIN_MINUTES = 30; MAX_TOTAL_STEPS = 0
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    NUM_ENVS = 16
    BATCH_SIZE = 512
    NUM_EPOCHS = 6
    MAX_TRAIN_MINUTES = 2
    MAX_TOTAL_STEPS = 0            # time-only for MPS
    print("✔ Apple MPS detected")
else:
    DEVICE = torch.device("cpu")
    NUM_ENVS = 4
    BATCH_SIZE = 256
    NUM_EPOCHS = 4
    MAX_TRAIN_MINUTES = 30         # 30 min for CPU
    MAX_TOTAL_STEPS = 0            # time-only for CPU
    print("⚠ CPU mode (no GPU)")
    if IS_KAGGLE:
        print("  ⚠ Kaggle detected but NO GPU!")
        print("  → Go to Settings → Accelerator → GPU P100")
        print("  → Then restart the notebook session")

if MAX_TOTAL_STEPS > 0 and MAX_TRAIN_MINUTES > 0:
    _limit_str = f"{MAX_TOTAL_STEPS/1e6:.0f}M steps OR {MAX_TRAIN_MINUTES} min"
elif MAX_TOTAL_STEPS > 0:
    _limit_str = f"{MAX_TOTAL_STEPS/1e6:.0f}M steps"
else:
    _limit_str = f"{MAX_TRAIN_MINUTES} min"
print(f"Device: {DEVICE}  |  Envs: {NUM_ENVS}  |  Batch: {BATCH_SIZE}  |  "
      f"Train limit: {_limit_str}")

In [ ]:
@dataclass
class Config:
    """All hyperparameters in one place."""
    arena_width: int = 30
    arena_height: int = 30
    num_hiders: int = 2
    num_seekers: int = 2
    num_boxes: int = 5
    max_episode_steps: int = 480
    box_width: float = 1.5
    box_height: float = 1.5
    agent_radius: float = 0.4
    grab_range: float = 1.5
    num_lidar_rays: int = 9
    lidar_range: float = 10.0
    local_obs_radius: float = 6.0
    max_nearby_entities: int = 6
    lr: float = 3e-4
    gamma: float = 0.99
    gae_lambda: float = 0.95
    clip_eps: float = 0.2
    entropy_coef: float = 0.008
    value_coef: float = 0.5
    max_grad_norm: float = 0.5
    num_envs: int = NUM_ENVS
    batch_size: int = BATCH_SIZE
    num_epochs: int = NUM_EPOCHS
    rollout_steps: int = 128
    max_train_minutes: float = MAX_TRAIN_MINUTES
    max_total_steps: int = MAX_TOTAL_STEPS
    pool_size: int = 20
    pool_update_interval: int = 50
    save_interval: int = 50
    checkpoint_path: str = "data/checkpoint.pth"
    # ── Reward tuning (v15 — push hider win 65%→50%) ──
    # Per-step rewards SMALL (accumulate over 480 steps!)
    see_reward: float = 0.08          # seeker sees a hider
    unseen_reward: float = 0.08       # hider stays hidden
    chase_reward: float = 0.03        # seeker always-on gradient toward hiders
    pursuit_reward: float = 0.15      # seeker visibility-gated approach bonus
    evasion_reward: float = 0.15      # hider rewarded for distance when seen
    spread_reward: float = 0.05       # seekers fan out
    idle_penalty: float = -0.3        # standing still punishment
    idle_threshold: int = 8
    # Terminal rewards BIG (fire once per episode)
    catch_radius: float = 4.5         # catch zone (wider = easier to catch)
    catch_bonus: float = 14.0         # seeker catches hider
    survival_bonus: float = 3.5       # ALL hiders survive (2×3.5=7 << 14 catch)
    unseen_bonus: float = 1.0         # extra for never-seen hiders
    seeker_fail_penalty: float = -8.0  # seekers fail to catch anyone

config = Config()
_limit = f"{config.max_train_minutes:.0f} min"
if config.max_total_steps > 0:
    _limit += f" OR {config.max_total_steps/1e6:.0f}M steps"
print(f"Config  {config.num_hiders}v{config.num_seekers}  |  "
      f"Arena {config.arena_width}x{config.arena_height}  |  "
      f"{config.num_boxes} boxes  |  {config.max_episode_steps} steps/ep  |  "
      f"Train: {_limit}  |  entropy={config.entropy_coef}")
print(f"  chase={config.chase_reward}  pursuit={config.pursuit_reward}  "
      f"catch_bonus={config.catch_bonus}  catch_r={config.catch_radius}  "
      f"survival={config.survival_bonus}  evasion={config.evasion_reward}  "
      f"fail={config.seeker_fail_penalty}  idle={config.idle_penalty}")
print(f"  Win condition: hiders win if NOT caught (timeout = hider win)")

In [ ]:
class PPONetwork(nn.Module):
    """Actor-Critic with shared backbone.
    Input -> Linear(256) -> LN -> ReLU -> Linear(256) -> LN -> ReLU
          -> Policy Head (logits)  |  Value Head (scalar)
    """
    def __init__(self, obs_dim: int, act_dim: int) -> None:
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(obs_dim, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
        )
        self.policy_head = nn.Linear(256, act_dim)
        self.value_head = nn.Linear(256, 1)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.orthogonal_(m.weight, gain=np.sqrt(2))
                nn.init.zeros_(m.bias)
        nn.init.orthogonal_(self.policy_head.weight, gain=0.01)
        nn.init.zeros_(self.policy_head.bias)
    def forward(self, obs: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        feat = self.backbone(obs)
        return self.policy_head(feat), self.value_head(feat).squeeze(-1)
    def get_action(self, obs: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        logits, value = self.forward(obs)
        dist = torch.distributions.Categorical(logits=logits)
        action = dist.sample()
        return action, dist.log_prob(action), value
    def evaluate_actions(self, obs: torch.Tensor, actions: torch.Tensor
                         ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        logits, value = self.forward(obs)
        dist = torch.distributions.Categorical(logits=logits)
        return dist.log_prob(actions), value, dist.entropy()
_test_net = PPONetwork(40, 6)
print(f"PPONetwork  params: {sum(p.numel() for p in _test_net.parameters()):,}")
del _test_net

In [ ]:
class RolloutBuffer:
    """Stores trajectories and computes GAE for PPO."""
    def __init__(self, num_steps: int, num_envs: int,
                 num_agents: int, obs_dim: int) -> None:
        self.num_steps = num_steps
        self.num_envs = num_envs
        self.num_agents = num_agents
        self.obs_dim = obs_dim
        self.reset()
    def reset(self) -> None:
        S, E, A, D = self.num_steps, self.num_envs, self.num_agents, self.obs_dim
        self.observations = np.zeros((S, E, A, D), dtype=np.float32)
        self.actions      = np.zeros((S, E, A),    dtype=np.int64)
        self.log_probs    = np.zeros((S, E, A),    dtype=np.float32)
        self.rewards      = np.zeros((S, E, A),    dtype=np.float32)
        self.values       = np.zeros((S, E, A),    dtype=np.float32)
        self.dones        = np.zeros((S, E),       dtype=np.float32)
        self.ptr = 0
    def add(self, obs: np.ndarray, actions: np.ndarray,
            log_probs: np.ndarray, rewards: np.ndarray,
            values: np.ndarray, dones: np.ndarray) -> None:
        self.observations[self.ptr] = obs
        self.actions[self.ptr]      = actions
        self.log_probs[self.ptr]    = log_probs
        self.rewards[self.ptr]      = rewards
        self.values[self.ptr]       = values
        self.dones[self.ptr]        = dones
        self.ptr += 1
    def compute_gae(self, last_values: np.ndarray,
                    gamma: float, lam: float
                    ) -> Tuple[np.ndarray, np.ndarray]:
        """Generalized Advantage Estimation."""
        advantages = np.zeros_like(self.rewards)
        last_gae = np.zeros((self.num_envs, self.num_agents), dtype=np.float32)
        for t in reversed(range(self.num_steps)):
            next_val = last_values if t == self.num_steps - 1 else self.values[t + 1]
            non_term = (1.0 - self.dones[t])[:, np.newaxis]   
            delta = self.rewards[t] + gamma * next_val * non_term - self.values[t]
            last_gae = delta + gamma * lam * non_term * last_gae
            advantages[t] = last_gae
        returns = advantages + self.values
        return advantages, returns
    def get_batches(self, advantages: np.ndarray, returns: np.ndarray,
                    batch_size: int):
        """Yield shuffled mini-batches."""
        total = self.num_steps * self.num_envs * self.num_agents
        obs_flat  = self.observations.reshape(total, -1)
        act_flat  = self.actions.reshape(total)
        lp_flat   = self.log_probs.reshape(total)
        adv_flat  = advantages.reshape(total)
        ret_flat  = returns.reshape(total)
        adv_flat = (adv_flat - adv_flat.mean()) / (adv_flat.std() + 1e-8)
        indices = np.random.permutation(total)
        for start in range(0, total, batch_size):
            idx = indices[start:start + batch_size]
            yield (
                torch.tensor(obs_flat[idx],  device=DEVICE),
                torch.tensor(act_flat[idx],  device=DEVICE),
                torch.tensor(lp_flat[idx],   device=DEVICE),
                torch.tensor(adv_flat[idx],  device=DEVICE),
                torch.tensor(ret_flat[idx],  device=DEVICE),
            )
print("RolloutBuffer ready")

In [ ]:
class Arena:
    """NumPy-vectorized 2D Hide & Seek with static walls.
    Actions: 0=noop  1=up  2=down  3=left  4=right  5=grab/release
    """
    NUM_ACTIONS: int = 6
    MOVE_SPEED: float = 0.5
    def __init__(self, cfg: Config) -> None:
        self.cfg = cfg
        self.NE = cfg.num_envs
        self.NH = cfg.num_hiders
        self.NS = cfg.num_seekers
        self.NA = self.NH + self.NS
        self.NB = cfg.num_boxes
        self.W  = float(cfg.arena_width)
        self.H  = float(cfg.arena_height)
        self.lidar_angles = np.linspace(
            -np.pi * 0.75, np.pi * 0.75, cfg.num_lidar_rays, dtype=np.float32)
        self.obs_dim = (cfg.num_lidar_rays + 2 + 2 + 1
                        + cfg.max_nearby_entities * 4 + 1 + 1)
        self.agent_pos     = np.zeros((self.NE, self.NA, 2), np.float32)
        self.agent_vel     = np.zeros((self.NE, self.NA, 2), np.float32)
        self.agent_heading = np.zeros((self.NE, self.NA),    np.float32)
        self.agent_grabbed = np.full((self.NE, self.NA), -1, np.int32)
        self.agent_idle    = np.zeros((self.NE, self.NA),    np.int32)
        self.box_pos  = np.zeros((self.NE, self.NB, 2), np.float32)
        self.box_size = np.broadcast_to(
            np.array([cfg.box_width, cfg.box_height], np.float32),
            (self.NE, self.NB, 2)).copy()
        self.box_grabbed_by = np.full((self.NE, self.NB), -1, np.int32)
        self.step_count              = np.zeros(self.NE, np.int32)
        self.hider_seen              = np.zeros((self.NE, self.NH), bool)
        self.episode_hider_ever_seen = np.zeros((self.NE, self.NH), bool)
        # ── Static walls ──
        T = 0.6
        W, H = self.W, self.H
        wall_defs = [
            # ── Center cross (shorter for wider passages) ──
            (W*0.50, H*0.50, W*0.28, T),      # center H
            (W*0.50, H*0.50, T,      H*0.28), # center V
            # ── Corner alcoves (L-shapes, trimmed) ──
            (W*0.18, H*0.84, W*0.18, T),      # TL H
            (W*0.08, H*0.74, T,      H*0.16), # TL V
            (W*0.82, H*0.84, W*0.18, T),      # TR H
            (W*0.92, H*0.74, T,      H*0.16), # TR V
            (W*0.18, H*0.16, W*0.18, T),      # BL H
            (W*0.08, H*0.26, T,      H*0.16), # BL V
            (W*0.82, H*0.16, W*0.18, T),      # BR H
            (W*0.92, H*0.26, T,      H*0.16), # BR V
            # ── Mid-field barriers (narrower) ──
            (W*0.30, H*0.70, W*0.10, T),      # upper-left H
            (W*0.70, H*0.30, W*0.10, T),      # lower-right H
            # ── Extra hiding spots ──
            (W*0.35, H*0.88, T,      H*0.10), # top-center-left V
            (W*0.65, H*0.12, T,      H*0.10), # bottom-center-right V
            (W*0.13, H*0.50, W*0.08, T),      # mid-left H
            (W*0.87, H*0.50, W*0.08, T),      # mid-right H
            # ── Additional cover (moved further from center) ──
            (W*0.33, H*0.35, T,      H*0.10), # inner-left V
            (W*0.67, H*0.65, T,      H*0.10), # inner-right V
            (W*0.22, H*0.56, W*0.08, T),      # left-mid H
            (W*0.78, H*0.44, W*0.08, T),      # right-mid H
        ]
        self.NW = len(wall_defs)
        self.wall_pos  = np.array([[d[0], d[1]] for d in wall_defs], np.float32)
        self.wall_size = np.array([[d[2], d[3]] for d in wall_defs], np.float32)
        self.wall_min  = self.wall_pos - self.wall_size * 0.5
        self.wall_max  = self.wall_pos + self.wall_size * 0.5
        self._diag = np.sqrt(self.W**2 + self.H**2)
        self.reset_all()
    # ── Reset ──
    def reset_all(self) -> np.ndarray:
        return self._reset_envs(np.arange(self.NE))
    def _reset_envs(self, idx: np.ndarray) -> np.ndarray:
        n = len(idx); m = 2.0
        self.agent_pos[idx] = np.random.uniform(
            [m,m],[self.W-m,self.H-m],(n,self.NA,2)).astype(np.float32)
        self.agent_vel[idx] = 0
        self.agent_heading[idx] = np.random.uniform(
            -np.pi,np.pi,(n,self.NA)).astype(np.float32)
        self.agent_grabbed[idx] = -1; self.agent_idle[idx] = 0
        self.box_pos[idx] = np.random.uniform(
            [m,m],[self.W-m,self.H-m],(n,self.NB,2)).astype(np.float32)
        self.box_grabbed_by[idx] = -1; self.step_count[idx] = 0
        self.hider_seen[idx] = False; self.episode_hider_ever_seen[idx] = False
        r = self.cfg.agent_radius
        for w in range(self.NW):
            half = self.wall_size[w]*0.5 + r
            rel = self.agent_pos[idx] - self.wall_pos[w]
            ox = half[0]-np.abs(rel[:,:,0]); oy = half[1]-np.abs(rel[:,:,1])
            c = (ox>0)&(oy>0); px = ox<oy
            self.agent_pos[idx,:,0] += np.where(c&px, np.sign(rel[:,:,0])*ox, 0)
            self.agent_pos[idx,:,1] += np.where(c&~px, np.sign(rel[:,:,1])*oy, 0)
        for w in range(self.NW):
            wh = self.wall_size[w]*0.5
            for bi in range(self.NB):
                bh = self.box_size[0,bi]*0.5; th = wh+bh
                rel = self.box_pos[idx,bi] - self.wall_pos[w]
                ox = th[0]-np.abs(rel[:,0]); oy = th[1]-np.abs(rel[:,1])
                c = (ox>0)&(oy>0); px = ox<oy
                self.box_pos[idx,bi,0] += np.where(c&px, np.sign(rel[:,0])*ox, 0)
                self.box_pos[idx,bi,1] += np.where(c&~px, np.sign(rel[:,1])*oy, 0)
        return self._observe()
    # ── Observation ──
    def _observe(self) -> np.ndarray:
        obs = np.zeros((self.NE, self.NA, self.obs_dim), np.float32)
        lidar = self._cast_lidar()
        team = np.zeros((self.NE,self.NA,1), np.float32); team[:,self.NH:,0]=1.0
        nearby = self._nearby_entities()
        grabbed = (self.agent_grabbed>=0).astype(np.float32)[:,:,np.newaxis]
        norm_p = self.agent_pos / np.array([self.W,self.H], np.float32)
        frac = (self.step_count[:,np.newaxis,np.newaxis]
                / self.cfg.max_episode_steps
                * np.ones((1,self.NA,1), np.float32))
        i=0; NR=self.cfg.num_lidar_rays
        obs[:,:,i:i+NR]=lidar; i+=NR
        obs[:,:,i:i+2]=norm_p; i+=2
        obs[:,:,i:i+2]=self.agent_vel; i+=2
        obs[:,:,i:i+1]=team; i+=1
        K4=self.cfg.max_nearby_entities*4
        obs[:,:,i:i+K4]=nearby; i+=K4
        obs[:,:,i:i+1]=grabbed; i+=1
        obs[:,:,i:i+1]=frac; i+=1
        return obs
    # ── LiDAR ──
    def _cast_lidar(self) -> np.ndarray:
        NR, rng = self.cfg.num_lidar_rays, self.cfg.lidar_range
        angles = self.agent_heading[:,:,np.newaxis] + self.lidar_angles[np.newaxis,np.newaxis,:]
        dx, dy = np.cos(angles), np.sin(angles)
        dist = np.full_like(dx, rng)
        px, py = self.agent_pos[:,:,0:1], self.agent_pos[:,:,1:2]
        with np.errstate(divide='ignore', invalid='ignore'):
            dist=np.minimum(dist,np.clip(np.where(dx> 1e-6,(self.W-px)/dx,rng),0,rng))
            dist=np.minimum(dist,np.clip(np.where(dx<-1e-6,-px/dx,rng),0,rng))
            dist=np.minimum(dist,np.clip(np.where(dy> 1e-6,(self.H-py)/dy,rng),0,rng))
            dist=np.minimum(dist,np.clip(np.where(dy<-1e-6,-py/dy,rng),0,rng))
        with np.errstate(divide='ignore', invalid='ignore'):
            inv_dx=np.where(np.abs(dx)>1e-8,1.0/dx,1e8)
            inv_dy=np.where(np.abs(dy)>1e-8,1.0/dy,1e8)
        bmin=self.box_pos-self.box_size*0.5; bmax=self.box_pos+self.box_size*0.5
        for b in range(self.NB):
            bx0=bmin[:,b,0][:,np.newaxis,np.newaxis]; by0=bmin[:,b,1][:,np.newaxis,np.newaxis]
            bx1=bmax[:,b,0][:,np.newaxis,np.newaxis]; by1=bmax[:,b,1][:,np.newaxis,np.newaxis]
            t1x=(bx0-px)*inv_dx; t2x=(bx1-px)*inv_dx
            t1y=(by0-py)*inv_dy; t2y=(by1-py)*inv_dy
            tmn=np.maximum(np.minimum(t1x,t2x),np.minimum(t1y,t2y))
            tmx=np.minimum(np.maximum(t1x,t2x),np.maximum(t1y,t2y))
            hit=(tmx>=np.maximum(tmn,0.0))&(tmn<dist)
            dist=np.where(hit,np.minimum(dist,np.maximum(tmn,0.0)),dist)
        for w in range(self.NW):
            wx0,wy0=self.wall_min[w]; wx1,wy1=self.wall_max[w]
            t1x=(wx0-px)*inv_dx; t2x=(wx1-px)*inv_dx
            t1y=(wy0-py)*inv_dy; t2y=(wy1-py)*inv_dy
            tmn=np.maximum(np.minimum(t1x,t2x),np.minimum(t1y,t2y))
            tmx=np.minimum(np.maximum(t1x,t2x),np.maximum(t1y,t2y))
            hit=(tmx>=np.maximum(tmn,0.0))&(tmn<dist)
            dist=np.where(hit,np.minimum(dist,np.maximum(tmn,0.0)),dist)
        return dist/rng
    # ── Nearby entities ──
    def _nearby_entities(self) -> np.ndarray:
        K, rad = self.cfg.max_nearby_entities, self.cfg.local_obs_radius
        out = np.zeros((self.NE,self.NA,K*4), np.float32)
        rel = self.agent_pos[:,np.newaxis,:,:] - self.agent_pos[:,:,np.newaxis,:]
        d = np.linalg.norm(rel, axis=-1)
        d = np.where(np.eye(self.NA,dtype=bool)[np.newaxis], rad+1, d)
        te = np.zeros(self.NA, np.float32); te[self.NH:]=1.0
        ne=np.arange(self.NE)[:,np.newaxis]; na=np.arange(self.NA)[np.newaxis,:]
        for k in range(min(K, self.NA-1)):
            ci=np.argmin(d,axis=-1); cd=d[ne,na,ci]; ok=cd<rad
            rp=rel[ne,na,ci]; b=k*4
            out[:,:,b]=np.where(ok,rp[:,:,0]/rad,0)
            out[:,:,b+1]=np.where(ok,rp[:,:,1]/rad,0)
            out[:,:,b+2]=np.where(ok,1.0,0)
            out[:,:,b+3]=np.where(ok,te[ci],0)
            d[ne,na,ci]=rad+1
        return out
    # ── Line of sight ──
    def _check_los(self) -> np.ndarray:
        hpos=self.agent_pos[:,:self.NH]; spos=self.agent_pos[:,self.NH:]
        bmin=self.box_pos-self.box_size*0.5; bmax=self.box_pos+self.box_size*0.5
        visible = np.zeros((self.NE,self.NH), bool)
        for s in range(self.NS):
            sp=spos[:,s:s+1,:]; direction=hpos-sp
            length=np.linalg.norm(direction,axis=-1,keepdims=True)+1e-8
            dn=direction/length
            with np.errstate(divide='ignore',invalid='ignore'):
                inv=np.where(np.abs(dn)>1e-8,1.0/dn,1e8)
            blocked=np.zeros((self.NE,self.NH),bool)
            for b in range(self.NB):
                t1=(bmin[:,b:b+1,:]-sp)*inv; t2=(bmax[:,b:b+1,:]-sp)*inv
                tmn=np.maximum(np.minimum(t1,t2).max(axis=-1),0.0)
                tmx=np.minimum(np.maximum(t1,t2).min(axis=-1),length.squeeze(-1))
                blocked|=(tmx>tmn)
            for w in range(self.NW):
                t1=(self.wall_min[w]-sp)*inv; t2=(self.wall_max[w]-sp)*inv
                tmn=np.maximum(np.minimum(t1,t2).max(axis=-1),0.0)
                tmx=np.minimum(np.maximum(t1,t2).min(axis=-1),length.squeeze(-1))
                blocked|=(tmx>tmn)
            visible|=~blocked
        return visible
    # ── Step ──
    def step(self, actions: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray, Dict]:
        prev = self.agent_pos.copy()
        dx=np.zeros((self.NE,self.NA),np.float32); dy=np.zeros_like(dx)
        dx[actions==3]=-self.MOVE_SPEED; dx[actions==4]=self.MOVE_SPEED
        dy[actions==1]=self.MOVE_SPEED;  dy[actions==2]=-self.MOVE_SPEED
        delta=np.stack([dx,dy],axis=-1)
        moving=np.linalg.norm(delta,axis=-1)>0.01
        self.agent_heading=np.where(moving,np.arctan2(dy,dx),self.agent_heading)
        new_pos=self.agent_pos+delta; r=self.cfg.agent_radius
        new_pos[:,:,0]=np.clip(new_pos[:,:,0],r,self.W-r)
        new_pos[:,:,1]=np.clip(new_pos[:,:,1],r,self.H-r)
        for b in range(self.NB):
            bp=self.box_pos[:,b:b+1,:]; half=self.box_size[:,b:b+1,:]*0.5+r
            rel=new_pos-bp; ox=half[:,:,0:1]-np.abs(rel[:,:,0:1]); oy=half[:,:,1:2]-np.abs(rel[:,:,1:2])
            c=(ox>0)&(oy>0); px=ox<oy
            new_pos[:,:,0:1]+=np.where(c&px,np.sign(rel[:,:,0:1])*ox,0)
            new_pos[:,:,1:2]+=np.where(c&~px,np.sign(rel[:,:,1:2])*oy,0)
        for w in range(self.NW):
            hw=self.wall_size[w]*0.5+r; rel=new_pos-self.wall_pos[w]
            ox=hw[0]-np.abs(rel[:,:,0]); oy=hw[1]-np.abs(rel[:,:,1])
            c=(ox>0)&(oy>0); px=ox<oy
            new_pos[:,:,0]+=np.where(c&px,np.sign(rel[:,:,0])*ox,0)
            new_pos[:,:,1]+=np.where(c&~px,np.sign(rel[:,:,1])*oy,0)
        self.agent_pos=new_pos; self.agent_vel=self.agent_pos-prev
        grab_mask=(actions==5)
        for b in range(self.NB):
            d2b=np.linalg.norm(self.agent_pos-self.box_pos[:,b:b+1,:],axis=-1)
            holding=(self.agent_grabbed==b)&grab_mask
            self.agent_grabbed=np.where(holding,-1,self.agent_grabbed)
            self.box_grabbed_by[:,b]=np.where(holding.any(axis=-1),-1,self.box_grabbed_by[:,b])
            can=(d2b<self.cfg.grab_range)&grab_mask&(self.agent_grabbed==-1)&(self.box_grabbed_by[:,b:b+1]==-1)
            for a in range(self.NA):
                ok=can[:,a]&(self.box_grabbed_by[:,b]==-1)
                self.agent_grabbed[:,a]=np.where(ok,b,self.agent_grabbed[:,a])
                self.box_grabbed_by[:,b]=np.where(ok,a,self.box_grabbed_by[:,b])
        for b in range(self.NB):
            g=self.box_grabbed_by[:,b]; held=g>=0
            if held.any():
                for a in range(self.NA):
                    m=held&(g==a)
                    if m.any(): self.box_pos[m,b]+=self.agent_vel[m,a]
                hs=self.box_size[:,b,:]*0.5
                self.box_pos[:,b,0]=np.clip(self.box_pos[:,b,0],hs[:,0],self.W-hs[:,0])
                self.box_pos[:,b,1]=np.clip(self.box_pos[:,b,1],hs[:,1],self.H-hs[:,1])
                for w in range(self.NW):
                    wh=self.wall_size[w]*0.5; bh=hs; th=wh+bh
                    rel=self.box_pos[:,b]-self.wall_pos[w]
                    ox=th[:,0]-np.abs(rel[:,0]); oy=th[:,1]-np.abs(rel[:,1])
                    c=(ox>0)&(oy>0); px=ox<oy
                    self.box_pos[:,b,0]+=np.where(c&px,np.sign(rel[:,0])*ox,0)
                    self.box_pos[:,b,1]+=np.where(c&~px,np.sign(rel[:,1])*oy,0)
        active=(np.linalg.norm(self.agent_vel,axis=-1)>0.01)|(actions!=0)
        self.agent_idle=np.where(active,0,self.agent_idle+1)
        self.step_count+=1
        # ── Visibility & Rewards ──
        vis=self._check_los(); self.hider_seen=vis; self.episode_hider_ever_seen|=vis
        rewards=np.zeros((self.NE,self.NA),np.float32)
        any_seen=vis.any(axis=-1)
        # Seeker: reward for seeing, penalty for not seeing
        rewards[:,self.NH:]=np.where(any_seen[:,np.newaxis],self.cfg.see_reward,-self.cfg.see_reward)
        # Hider: penalty for being seen, reward for staying hidden
        rewards[:,:self.NH]=np.where(vis,-self.cfg.unseen_reward,self.cfg.unseen_reward)
        # ── Pursuit / Evasion / Catch ──
        _hp = self.agent_pos[:, :self.NH]
        _sp = self.agent_pos[:, self.NH:]
        catch_dones = np.zeros(self.NE, dtype=bool)
        for h_i in range(self.NH):
            for s_i in range(self.NS):
                _dsh = np.linalg.norm(_hp[:, h_i] - _sp[:, s_i], axis=-1)
                _v = vis[:, h_i]
                # Chase: tiny always-on gradient toward hiders
                rewards[:, self.NH+s_i] += self.cfg.chase_reward*(1.0 - _dsh/self._diag)
                # Pursuit: visibility-gated approach bonus
                rewards[:, self.NH+s_i] += self.cfg.pursuit_reward*(1.0 - _dsh/self._diag)*_v
                # Evasion: hider rewarded for distance when visible
                rewards[:, h_i] += self.cfg.evasion_reward*(_dsh/self._diag)*_v
                # Catch: seeker within catch_radius of visible hider = tagged!
                _caught = (_dsh < self.cfg.catch_radius) & _v
                catch_dones |= _caught
                rewards[:, self.NH+s_i] += np.where(_caught, self.cfg.catch_bonus, 0.0)
                rewards[:, h_i] += np.where(_caught, -self.cfg.catch_bonus, 0.0)
        # ── Smart idle penalty ──
        _idle = self.agent_idle > self.cfg.idle_threshold
        rewards[:, self.NH:] += np.where(_idle[:, self.NH:], self.cfg.idle_penalty, 0.0)
        rewards[:, :self.NH] += np.where(_idle[:, :self.NH] & vis, self.cfg.idle_penalty, 0.0)
        # ── Seeker spread reward ──
        if self.NS > 1:
            for si in range(self.NS):
                for sj in range(si+1, self.NS):
                    _dss = np.linalg.norm(_sp[:, si] - _sp[:, sj], axis=-1)
                    rewards[:, self.NH+si] += self.cfg.spread_reward*(_dss/self._diag)
                    rewards[:, self.NH+sj] += self.cfg.spread_reward*(_dss/self._diag)
        # ── Episode end ──
        time_dones = self.step_count >= self.cfg.max_episode_steps
        dones = time_dones | catch_dones
        if time_dones.any():
            di = np.where(time_dones)[0]
            # ALL hiders survived (not caught) — survival bonus for everyone
            rewards[di, :self.NH] += self.cfg.survival_bonus
            # Extra bonus for hiders who were never seen at all
            rewards[di, :self.NH] += (~self.episode_hider_ever_seen[di]) * self.cfg.unseen_bonus
            # Seekers failed to catch — penalty
            rewards[di, self.NH:] += self.cfg.seeker_fail_penalty
        obs=self._observe()
        info={"done_mask":dones.copy(),
              "hider_ever_seen": self.episode_hider_ever_seen.copy(),
              "catch_dones": catch_dones.copy()}
        if dones.any(): self._reset_envs(np.where(dones)[0])
        return obs, rewards, dones.astype(np.float32), info

_a = Arena(config)
print(f"Arena  obs_dim={_a.obs_dim}  envs={_a.NE}  walls={_a.NW}  boxes={_a.NB}  speed={Arena.MOVE_SPEED}")
del _a

In [ ]:
class SelfPlayPool:
    """Stores past policy snapshots for opponent sampling."""
    def __init__(self, max_size: int = 20) -> None:
        self.max_size = max_size
        self.policies: List[Dict[str, torch.Tensor]] = []
        self.win_rates: List[float] = []
    def add(self, state_dict: Dict[str, torch.Tensor],
            win_rate: float = 0.5) -> None:
        self.policies.append({k: v.cpu().clone() for k, v in state_dict.items()})
        self.win_rates.append(win_rate)
        if len(self.policies) > self.max_size:
            self.policies.pop(0)
            self.win_rates.pop(0)
    def sample(self) -> Optional[Dict[str, torch.Tensor]]:
        if not self.policies:
            return None
        return self.policies[np.random.randint(len(self.policies))]
    def __len__(self) -> int:
        return len(self.policies)
print("SelfPlayPool ready")

In [ ]:
class PPOTrainer:
    """Centralized Training, Decentralized Execution.
    Separate hider / seeker networks with parameter sharing within teams.
    """
    def __init__(self, cfg: Config, obs_dim: int) -> None:
        self.cfg = cfg
        self.obs_dim = obs_dim
        act_dim = Arena.NUM_ACTIONS
        self.hider_net  = PPONetwork(obs_dim, act_dim).to(DEVICE)
        self.seeker_net = PPONetwork(obs_dim, act_dim).to(DEVICE)
        self.hider_opt  = optim.Adam(self.hider_net.parameters(),  lr=cfg.lr, eps=1e-5)
        self.seeker_opt = optim.Adam(self.seeker_net.parameters(), lr=cfg.lr, eps=1e-5)
        self.pool = SelfPlayPool(cfg.pool_size)
        self.episode_count = 0
        self.total_steps   = 0
        self.hider_win_rates: List[float]  = []
        self.step_log: List[Tuple[int, float, float]] = []
        self.entropy_log: List[Tuple[float, float]] = []   # (hider_ent, seeker_ent)
    def collect_rollout(self, arena: Arena,
                        buf_h: RolloutBuffer,
                        buf_s: RolloutBuffer) -> Dict:
        buf_h.reset()
        buf_s.reset()
        obs = arena._observe()
        ep_rwd_h = np.zeros(arena.NE, np.float32)
        ep_rwd_s = np.zeros(arena.NE, np.float32)
        eps_done = 0
        h_wins   = 0
        for _ in range(self.cfg.rollout_steps):
            ot = torch.tensor(obs, device=DEVICE)
            with torch.no_grad():
                ho = ot[:, :self.cfg.num_hiders].reshape(-1, self.obs_dim)
                ha, hlp, hv = self.hider_net.get_action(ho)
                ha  = ha.reshape(arena.NE, self.cfg.num_hiders)
                hlp = hlp.reshape(arena.NE, self.cfg.num_hiders)
                hv  = hv.reshape(arena.NE, self.cfg.num_hiders)
                so = ot[:, self.cfg.num_hiders:].reshape(-1, self.obs_dim)
                sa, slp, sv = self.seeker_net.get_action(so)
                sa  = sa.reshape(arena.NE, self.cfg.num_seekers)
                slp = slp.reshape(arena.NE, self.cfg.num_seekers)
                sv  = sv.reshape(arena.NE, self.cfg.num_seekers)
            all_act = np.concatenate([ha.cpu().numpy(), sa.cpu().numpy()], axis=1)
            nobs, rwd, dones, info = arena.step(all_act)
            buf_h.add(obs[:, :self.cfg.num_hiders],
                      ha.cpu().numpy(), hlp.cpu().numpy(),
                      rwd[:, :self.cfg.num_hiders],
                      hv.cpu().numpy(), dones)
            buf_s.add(obs[:, self.cfg.num_hiders:],
                      sa.cpu().numpy(), slp.cpu().numpy(),
                      rwd[:, self.cfg.num_hiders:],
                      sv.cpu().numpy(), dones)
            ep_rwd_h += rwd[:, :self.cfg.num_hiders].mean(axis=1)
            ep_rwd_s += rwd[:, self.cfg.num_hiders:].mean(axis=1)
            if info["done_mask"].any():
                di = np.where(info["done_mask"])[0]
                eps_done += len(di)
                # Hiders win = survived (timeout), seekers win = catch
                h_wins += (~info["catch_dones"][di]).sum()
                ep_rwd_h[di] = 0
                ep_rwd_s[di] = 0
            obs = nobs
            self.total_steps += arena.NE
        self.step_log.append((self.total_steps, ep_rwd_h.mean().item(),
                              ep_rwd_s.mean().item()))
        with torch.no_grad():
            ot = torch.tensor(obs, device=DEVICE)
            _, hlv = self.hider_net(ot[:, :self.cfg.num_hiders].reshape(-1, self.obs_dim))
            hlv = hlv.reshape(arena.NE, self.cfg.num_hiders).cpu().numpy()
            _, slv = self.seeker_net(ot[:, self.cfg.num_hiders:].reshape(-1, self.obs_dim))
            slv = slv.reshape(arena.NE, self.cfg.num_seekers).cpu().numpy()
        hwr = h_wins / max(eps_done, 1)
        return {"hlv": hlv, "slv": slv, "eps": eps_done, "hwr": hwr}
    def ppo_update(self, buf: RolloutBuffer, net: PPONetwork,
                   opt: optim.Optimizer, last_v: np.ndarray
                   ) -> Dict[str, float]:
        adv, ret = buf.compute_gae(last_v, self.cfg.gamma, self.cfg.gae_lambda)
        pg_sum = vf_sum = ent_sum = 0.0
        n_up = 0
        for _ in range(self.cfg.num_epochs):
            for (ob, ac, olp, ad, rt) in buf.get_batches(adv, ret, self.cfg.batch_size):
                nlp, val, ent = net.evaluate_actions(ob, ac)
                ratio = torch.exp(nlp - olp)
                s1 = ratio * ad
                s2 = torch.clamp(ratio, 1 - self.cfg.clip_eps,
                                         1 + self.cfg.clip_eps) * ad
                pg_loss  = -torch.min(s1, s2).mean()
                vf_loss  = F.mse_loss(val, rt)
                ent_loss = -ent.mean()
                loss = (pg_loss
                        + self.cfg.value_coef * vf_loss
                        + self.cfg.entropy_coef * ent_loss)
                opt.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(net.parameters(), self.cfg.max_grad_norm)
                opt.step()
                pg_sum  += pg_loss.item()
                vf_sum  += vf_loss.item()
                ent_sum += ent.mean().item()
                n_up    += 1
        return {"pg": pg_sum/max(n_up,1),
                "vf": vf_sum/max(n_up,1),
                "ent": ent_sum/max(n_up,1)}
    def save(self, path: str) -> None:
        torch.save({
            "hider_net":       self.hider_net.state_dict(),
            "seeker_net":      self.seeker_net.state_dict(),
            "hider_optimizer": self.hider_opt.state_dict(),
            "seeker_optimizer":self.seeker_opt.state_dict(),
            "episode_count":   self.episode_count,
            "total_steps":     self.total_steps,
            "self_play_pool":  {"policies": self.pool.policies,
                                "win_rates": self.pool.win_rates},
            "hider_win_rates": self.hider_win_rates,
            "step_log":        self.step_log,
            "entropy_log":     self.entropy_log,
            "config":          self.cfg.__dict__,
        }, path)
        print(f"  \u2714 Saved  ep={self.episode_count}  steps={self.total_steps:,}")
    def load(self, path: str) -> bool:
        if not os.path.exists(path):
            return False
        ck = torch.load(path, map_location=DEVICE, weights_only=False)
        self.hider_net.load_state_dict(ck["hider_net"])
        self.seeker_net.load_state_dict(ck["seeker_net"])
        self.hider_opt.load_state_dict(ck["hider_optimizer"])
        self.seeker_opt.load_state_dict(ck["seeker_optimizer"])
        self.episode_count   = ck["episode_count"]
        self.total_steps     = ck["total_steps"]
        self.hider_win_rates = ck.get("hider_win_rates", [])
        self.step_log        = ck.get("step_log", [])
        self.entropy_log     = ck.get("entropy_log", [])
        if "self_play_pool" in ck:
            sp = ck["self_play_pool"]
            self.pool.policies  = sp["policies"]
            self.pool.win_rates = sp["win_rates"]
        print(f"  \u2714 Loaded  ep={self.episode_count}  steps={self.total_steps:,}")
        return True
print("PPOTrainer ready")

In [ ]:
arena   = Arena(config)
trainer = PPOTrainer(config, arena.obs_dim)
if os.path.exists(config.checkpoint_path):
    trainer.load(config.checkpoint_path)
    print(f"Resumed from episode {trainer.episode_count}")
else:
    print("Starting fresh training")
buf_h = RolloutBuffer(config.rollout_steps, config.num_envs,
                      config.num_hiders, arena.obs_dim)
buf_s = RolloutBuffer(config.rollout_steps, config.num_envs,
                      config.num_seekers, arena.obs_dim)
net_params = sum(p.numel() for p in trainer.hider_net.parameters())

# ── Display training limits ──
if config.max_total_steps > 0 and config.max_train_minutes == 0:
    _limit_str = f"{config.max_total_steps/1e6:.0f}M steps (no time limit)"
elif config.max_total_steps > 0:
    _limit_str = f"{config.max_train_minutes:.0f} min OR {config.max_total_steps/1e6:.0f}M steps"
else:
    _limit_str = f"{config.max_train_minutes:.0f} min"
print(f"\nTraining until: {_limit_str}  on {DEVICE}")
print(f"Teams     {config.num_hiders} hiders  vs  {config.num_seekers} seekers")
print(f"Envs      {config.num_envs}   |  Rollout {config.rollout_steps} steps")
print(f"Network   {net_params:,} params per team")
if config.max_total_steps > 0:
    print(f"Progress  {trainer.total_steps:,} / {config.max_total_steps:,} steps "
          f"({trainer.total_steps/config.max_total_steps*100:.1f}%)")
print("=" * 70)

t0 = time.time()
ep0 = trainer.episode_count
step0 = trainer.total_steps
max_time = config.max_train_minutes * 60

while True:
    elapsed = time.time() - t0
    if config.max_train_minutes > 0 and elapsed >= max_time:
        print(f"\n⏱ Time limit reached ({config.max_train_minutes:.0f} min)")
        break
    if config.max_total_steps > 0 and trainer.total_steps >= config.max_total_steps:
        print(f"\n🎯 Step limit reached ({config.max_total_steps/1e6:.0f}M steps)")
        break
    if (trainer.episode_count > 0
            and trainer.episode_count % config.pool_update_interval == 0):
        wr = trainer.hider_win_rates[-1] if trainer.hider_win_rates else 0.5
        trainer.pool.add(trainer.seeker_net.state_dict(), wr)
        opp = trainer.pool.sample()
        if opp is not None:
            trainer.seeker_net.load_state_dict(opp)
            print(f"  [Self-Play] opponent loaded  pool={len(trainer.pool)}")
    info = trainer.collect_rollout(arena, buf_h, buf_s)
    hs = trainer.ppo_update(buf_h, trainer.hider_net,  trainer.hider_opt,  info["hlv"])
    ss = trainer.ppo_update(buf_s, trainer.seeker_net, trainer.seeker_opt, info["slv"])
    trainer.episode_count += info["eps"]
    trainer.hider_win_rates.append(info["hwr"])
    trainer.entropy_log.append((hs['ent'], ss['ent']))
    elapsed = time.time() - t0
    done_ep = trainer.episode_count - ep0
    sps = (trainer.total_steps - step0) / max(elapsed, 1)

    if done_ep % 10 == 0 or done_ep <= 3:
        progress = ""
        if config.max_total_steps > 0:
            pct = trainer.total_steps / config.max_total_steps * 100
            progress = f"  [{pct:.1f}%]"
        time_str = f"{elapsed:.0f}s" if config.max_train_minutes == 0 else f"{max(max_time-elapsed,0):.0f}s left"
        print(f"Ep {trainer.episode_count:>5d} | "
              f"Steps {trainer.total_steps:>9,}{progress} | "
              f"H_win {info['hwr']:.2f} | "
              f"H_ent {hs['ent']:.3f}  S_ent {ss['ent']:.3f} | "
              f"{sps:,.0f} sps | "
              f"{time_str}")
    if trainer.episode_count % config.save_interval < max(info["eps"], 1):
        trainer.save(config.checkpoint_path)

trainer.save(config.checkpoint_path)
dt = time.time() - t0
print(f"\nDone  {trainer.episode_count} ep  {trainer.total_steps:,} steps  "
      f"{dt:.1f}s  ({(trainer.total_steps-step0)/dt:,.0f} sps)")

In [ ]:
BG      = '#0d1117'
TEXT    = '#c9d1d9'
GRID    = '#21262d'
ORANGE  = '#f0883e'
BLUE    = '#58a6ff'
GREEN   = '#3fb950'
RED     = '#f85149'
PURPLE  = '#bc8cff'
CYAN    = '#39d2c0'
YELLOW  = '#e3b341'
def fmt_steps(x: float, _) -> str:
    if x >= 1e6: return f"{x/1e6:.1f}M"
    if x >= 1e3: return f"{x/1e3:.0f}K"
    return str(int(x))
def smooth(y: np.ndarray, w: int = 15) -> np.ndarray:
    if len(y) < w: return y
    return np.convolve(y, np.ones(w)/w, mode='same')
fig, axes = plt.subplots(3, 2, figsize=(14, 14), facecolor=BG,
                          gridspec_kw={'height_ratios': [2, 2, 1.5]})
ax1 = fig.add_subplot(3, 1, 1)   # row 1 full width — rewards
ax2 = fig.add_subplot(3, 1, 2)   # row 2 full width — win rate
ax3_l = axes[2, 0]                # row 3 left  — seeker entropy
ax3_r = axes[2, 1]                # row 3 right — hider entropy
# Hide the placeholder axes from rows 1-2
for r in range(2):
    for c in range(2):
        axes[r, c].set_visible(False)
for ax in (ax1, ax2, ax3_l, ax3_r):
    ax.set_facecolor(BG)
    ax.tick_params(colors=TEXT)
    for spine in ('top', 'right'):
        ax.spines[spine].set_visible(False)
    for spine in ('bottom', 'left'):
        ax.spines[spine].set_color(GRID)
    ax.grid(True, color=GRID, alpha=0.3)
# ── Row 1: Rewards ──
if trainer.step_log:
    steps = np.array([r[0] for r in trainer.step_log])
    h_rwd = np.array([r[1] for r in trainer.step_log])
    s_rwd = np.array([r[2] for r in trainer.step_log])
    hs = smooth(h_rwd)
    ss = smooth(s_rwd)
    ax1.fill_between(steps, 0, hs, alpha=0.15, color=BLUE)
    ax1.fill_between(steps, 0, ss, alpha=0.15, color=RED)
    ax1.plot(steps, hs, color=BLUE, lw=2, label='Hider Avg Reward')
    ax1.plot(steps, ss, color=RED,  lw=2, label='Seeker Avg Reward')
    best_h = np.array([max(h_rwd[:i+1]) for i in range(len(h_rwd))])
    ax1.plot(steps, best_h, color=ORANGE, lw=1.5, ls='--', alpha=0.7, label='Best Hider')
    peak_i = np.argmax(hs)
    ax1.annotate(f'Peak {hs[peak_i]:.2f}',
                 xy=(steps[peak_i], hs[peak_i]),
                 xytext=(steps[peak_i], hs[peak_i] + abs(hs[peak_i])*0.3 + 0.5),
                 arrowprops=dict(arrowstyle='->', color=ORANGE, lw=2),
                 fontsize=10, color=ORANGE, fontweight='bold')
    ax1.set_title('Training Progress \u2014 Reward per Rollout', color=TEXT,
                  fontsize=14, fontweight='bold', pad=12)
    ax1.set_xlabel('Total Environment Steps', color=TEXT)
    ax1.set_ylabel('Average Reward', color=TEXT)
    ax1.xaxis.set_major_formatter(FuncFormatter(fmt_steps))
    ax1.legend(facecolor='#161b22', edgecolor=GRID, labelcolor=TEXT)
# ── Row 2: Win Rate ──
if trainer.hider_win_rates:
    wr = np.array(trainer.hider_win_rates)
    wr_x = np.linspace(steps[0] if len(steps) else 0,
                       steps[-1] if len(steps) else 1, len(wr))
    wrs = smooth(wr)
    ax2.fill_between(wr_x, 0, wrs, alpha=0.15, color=GREEN)
    ax2.plot(wr_x, wrs, color=GREEN, lw=2, label='Hider Win Rate')
    ax2.axhline(0.5, color=PURPLE, ls='--', alpha=0.5, label='50% baseline')
    ax2.set_ylim(-0.05, 1.05)
    ax2.set_title('Hider Win Rate', color=TEXT, fontsize=14, fontweight='bold', pad=12)
    ax2.set_xlabel('Total Environment Steps', color=TEXT)
    ax2.set_ylabel('Win Rate', color=TEXT)
    ax2.xaxis.set_major_formatter(FuncFormatter(fmt_steps))
    ax2.legend(facecolor='#161b22', edgecolor=GRID, labelcolor=TEXT)
# ── Row 3: Entropy (Seeker left, Hider right) ──
if trainer.entropy_log:
    h_ent = np.array([e[0] for e in trainer.entropy_log])
    s_ent = np.array([e[1] for e in trainer.entropy_log])
    ent_x = np.linspace(steps[0] if len(steps) else 0,
                        steps[-1] if len(steps) else 1, len(h_ent))
    max_ent = np.log(6)  # ln(6 actions) ≈ 1.791
    # Seeker entropy (left)
    ax3_l.fill_between(ent_x, 0, smooth(s_ent), alpha=0.15, color=RED)
    ax3_l.plot(ent_x, smooth(s_ent), color=RED, lw=2, label='Seeker Entropy')
    ax3_l.axhline(max_ent, color=GRID, ls=':', alpha=0.5, label=f'Max ({max_ent:.2f})')
    ax3_l.axhline(max_ent*0.5, color=PURPLE, ls='--', alpha=0.3, label='Ideal zone')
    ax3_l.set_ylim(0, max_ent * 1.1)
    ax3_l.set_title('Seeker Entropy', color=TEXT, fontsize=12, fontweight='bold', pad=8)
    ax3_l.set_xlabel('Total Steps', color=TEXT)
    ax3_l.set_ylabel('Entropy', color=TEXT)
    ax3_l.xaxis.set_major_formatter(FuncFormatter(fmt_steps))
    ax3_l.legend(facecolor='#161b22', edgecolor=GRID, labelcolor=TEXT, fontsize=8)
    # Hider entropy (right)
    ax3_r.fill_between(ent_x, 0, smooth(h_ent), alpha=0.15, color=BLUE)
    ax3_r.plot(ent_x, smooth(h_ent), color=BLUE, lw=2, label='Hider Entropy')
    ax3_r.axhline(max_ent, color=GRID, ls=':', alpha=0.5, label=f'Max ({max_ent:.2f})')
    ax3_r.axhline(max_ent*0.5, color=PURPLE, ls='--', alpha=0.3, label='Ideal zone')
    ax3_r.set_ylim(0, max_ent * 1.1)
    ax3_r.set_title('Hider Entropy', color=TEXT, fontsize=12, fontweight='bold', pad=8)
    ax3_r.set_xlabel('Total Steps', color=TEXT)
    ax3_r.set_ylabel('Entropy', color=TEXT)
    ax3_r.xaxis.set_major_formatter(FuncFormatter(fmt_steps))
    ax3_r.legend(facecolor='#161b22', edgecolor=GRID, labelcolor=TEXT, fontsize=8)
fig.text(0.5, 0.01,
         f"Episodes: {trainer.episode_count}  |  "
         f"Agents: {config.num_hiders + config.num_seekers} "
         f"({config.num_hiders}v{config.num_seekers})  |  "
         f"Device: {DEVICE}  |  Steps: {trainer.total_steps:,}",
         ha='center', color=TEXT, fontsize=9, style='italic')
plt.tight_layout(rect=[0, 0.03, 1, 1])
os.makedirs('data', exist_ok=True)
plt.savefig('data/training_progress.png', dpi=150, facecolor=BG, bbox_inches='tight')
plt.show()
print("Chart saved: training_progress.png")